## Loading Processed Dataset

In [ ]:
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from scipy.sparse import hstack, csr_matrix

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [3]:
df = pd.read_csv(r"./dataset/processed_dataset.csv")
df.head()

,review,sentiment,lemmatized_tokens
0,one of the other reviewers has mentioned that ...,positive,"['one', 'reviewer', 'mention', 'watch', 'oz', ..."
1,a wonderful little production. the filming tec...,positive,"['wonderful', 'little', 'production', 'film', ..."
2,i thought this was a wonderful way to spend ti...,positive,"['think', 'wonderful', 'way', 'spend', 'time',..."
3,basically there's a family where a little boy ...,negative,"['basically', 'family', 'little', 'boy', 'jake..."
4,"petter mattei's ""love in the time of money"" is...",positive,"['petter', 'mattei', 'love', 'time', 'money', ..."


## 2. Feature Extraction

In [ ]:
# [Text Feature Extraction] Applying TF-IDF => Converting text -> numerical features
df["Clean_Text"] = df["Lemmatized_Tokens"].apply(lambda x: " ".join(x))

tfidf_vectorizer = TfidfVectorizer(min_df=3, ngram_range=(1,2))

X_tfidf = tfidf_vectorizer.fit_transform(df["Clean_Text"])
X_tfidf.shape

In [ ]:
# [Numeric Feature Extraction]
numeric_columns = ["Released_Year", "Runtime", "Meta_score", "No_of_Votes", "Gross"]

scaler = MinMaxScaler()
X_num_scaled = scaler.fit_transform(df[numeric_columns])
X_num_scaled_sparse = csr_matrix(X_num_scaled)

# Separating X and y
threshold = df["IMDB_Rating"].median()
X = hstack([X_tfidf, X_num_scaled_sparse])
y = (df["IMDB_Rating"] > threshold).astype(int)

## 3. Data Splitting & Model Building

In [ ]:
# Splitting Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Models
lr = LogisticRegression(max_iter=1000)
nb = MultinomialNB()
svm = LinearSVC()

# Training
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)
svm.fit(X_train, y_train)

## 4. Model Evaluation

In [ ]:
# Testing
y_pred_lr = lr.predict(X_test)
y_pred_nb = nb.predict(X_test)
y_pred_svm = svm.predict(X_test)

# Calculating Scores
lr_acc = accuracy_score(y_test, y_pred_lr)
lr_pre = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)

nb_acc = accuracy_score(y_test, y_pred_nb)
nb_pre = precision_score(y_test, y_pred_nb)
nb_recall = recall_score(y_test, y_pred_nb)
nb_f1 = f1_score(y_test, y_pred_nb)

svm_acc = accuracy_score(y_test, y_pred_svm)
svm_pre = precision_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_f1 = f1_score(y_test, y_pred_svm)

# Printing
print("\nLogistic Regression Performance")
print(f"Accuracy: {lr_acc:.2f}")
print(f"Precision: {lr_pre:.2f}")
print(f"Recall: {lr_recall:.2f}")
print(f"F1: {lr_f1:.2f}")

print("\nNaive Bayes Performance")
print(f"Accuracy: {nb_acc:.2f}")
print(f"Precision: {nb_pre:.2f}")
print(f"Recall: {nb_recall:.2f}")
print(f"F1: {nb_f1:.2f}")

print("\nSVM Performance")
print(f"Accuracy: {svm_acc:.2f}")
print(f"Precision: {svm_pre:.2f}")
print(f"Recall: {svm_recall:.2f}")
print(f"F1: {svm_f1:.2f}")